# Custom feature functions

The features used in `isb` are 'classical features', in that they are derived from classic computer vision filters like the Gaussian blur and Sobel filter. They're chosen to capture things that are typically useful to separate phases in bio- or materials imaging, such as local-average colour, edges and texture (in electron microscopy these correlate with changes in elemental composition and morphology). They have lots of good properties, such as being interpretable, relatively cheap to compute, and inherently homogeneous. However, they can struggle to capture more complex semantic distinctions where different phases have similar colours, like the ['pore-back' effect](https://iopscience.iop.org/article/10.1149/1945-7111/ac7a68) in porous media, or polymorphism. 

The rise of deep-learning and 'feature foundation models' has lead to people trying to integrate features from deep neural networks into the trainable segmentation workflow. Examples of this are [ConvPaint](https://github.com/guiwitz/napari-convpaint) & [FeatureForest](https://github.com/juglab/featureforest) in bioimaging, and [HR-Dv2](https://github.com/tldr-group/HR-Dv2) & [Vulture](https://github.com/tldr-group/vulture) in materials imaging. These can capture complex semantic concepts and greatly improve segmentations, but have some drawbacks: they can be expensive to compute, difficult to interpret, only available in low-resolution, and [display positional biases](https://arxiv.org/abs/2603.16840). 

One key benefit of supplementing trainable segmentation with these deep features is that they're completely 'drop-in' - the workflow stays exactly the same, it's just that every pixel now has more features that describe it. This means stuff like the classifier and post-processing can remain as-is. Because this is a pattern I use frequently, I made sure `isb` supported custom feature functions - this can be additional deep features, but could also be stuff like extra elemental information (i.e. from EDX). 

In [ ]:
import numpy as np
from interactive_seg_backend.file_handling import load_image, load_labels

image = load_image("../tests/data/4.jpg")
labels = load_labels("../tests/data/4_labels.tif")

You're going to need to install `vulture` and download the checkpoints for the following to work. From the root directory, run

```bash
git clone http://github.com/tldr-group/vulture
uv pip install vulture

mdkir trained_models/
chmod +x examples/download_chkpoints.sh
./examples/download_chkpoints.sh
```

Let's define our custom feature function that extracts (upsampled) features from DINOv2: 

In [ ]:
from interactive_seg_backend import FeatureConfig
from torch import no_grad
from vulture.main import CompleteUpsampler

upsampler = CompleteUpsampler("FEATUP", "../trained_models/fit_reg_f32.pth", device="cuda:0", to_half=True, to_eval=True, add_flash_attn=False)

def get_deep_features(image: np.ndarray, _feat_cfg: FeatureConfig) -> np.ndarray:
    with no_grad():
        hr_feats = upsampler.forward(image)
    hr_feats_np = hr_feats.squeeze(0).cpu().numpy()
    # pytorch is (B, C, H, W) and we want (H, W, C)
    hr_feats_np = hr_feats_np.transpose(1, 2, 0)
    return hr_feats_np

In [ ]:
from interactive_seg_backend import featurise, train_and_apply, FeatureConfig, TrainingConfig

feat_cfg = FeatureConfig()
train_cfg = TrainingConfig(feature_config=feat_cfg)

classical_feats = featurise(image, train_cfg)
classical_and_deep_feats = featurise(image, train_cfg, custom_fns=[(get_deep_features, False)])

In [ ]:
classical_seg, _, _ = train_and_apply(classical_feats, labels, train_cfg)
deep_seg, _, _ = train_and_apply(classical_and_deep_feats, labels, train_cfg)

Let's also extract the deep features so we can visualise them later! We'll do the visualisation via [PCA](https://en.wikipedia.org/wiki/Principal_component_analysis). This is PCA'ing the features from (H,W,C) channels -> (H,W,3) and then plotting it as an RGB image, such that similar colours correspond to similar vectors in feature-space. Note that because of various boring technical reasons, the deep features of this kind of model are already PCA'd, so we just need to take the first 3 channels. For the classical features we have to PCA them ourselves. 

(note: PCA vis works well for deep features, but is a bit misleading for classical features, as it tends to get skewed by frequency - lots of our classical features are edge detectors, so they tend to explain most the variance)

In [ ]:
from vulture.utils import do_2D_pca
h, w, c = classical_feats.shape
deep_feats = classical_and_deep_feats[:, :, c:]

min_, max_ = np.amin(deep_feats, axis=(0, 1)), np.amax(deep_feats, axis=(0, 1))
to_vis_deep = ((deep_feats - min_) / (max_ - min_ + 1e-8))[:, :, :3]

In [ ]:
classical_feats_tr = classical_feats.transpose(2, 0, 1)
to_vis_classical = do_2D_pca(classical_feats_tr, n_components=3, pre_norm='std', post_norm='minmax')

We can now plot and compare the features and their resulting segmentations. We've got another three phase material - [a cast iron alloy with graphite inclusions from the DoITPoMs library](https://www.doitpoms.ac.uk/miclib/full_record.php?id=394). The classical features struggle to distinguish between the inclusions and the background because of their similar colour, but the deep features capture the distinction well and leads to a much better segmentation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from interactive_seg_backend.utils import apply_labels_as_overlay, recolor_labels, PALETTE_RGB_NORM


N_ROWS, N_COLS = 3, 2
fw, fh = 4, 2
fig = plt.figure(figsize=(N_COLS * fw, N_ROWS * fh))
gs = gridspec.GridSpec(N_ROWS, N_COLS, figure=fig)

img_with_labels = apply_labels_as_overlay(labels, image, PALETTE_RGB_NORM[1:])

img_ax = fig.add_subplot(gs[0, 0:2])

img_ax.imshow(img_with_labels)
img_ax.set_title("Image + labels")
img_ax.axis('off')

classical_seg_recolored = recolor_labels(classical_seg + 1, PALETTE_RGB_NORM[1:])
deep_seg_recolored = recolor_labels(deep_seg + 1, PALETTE_RGB_NORM[1:])

arrs = [[to_vis_classical, classical_seg_recolored], [to_vis_deep, deep_seg_recolored]]
titles = [["Classical feats PCA", "Classical feats seg"], ["Deep feats PCA", "Classical + deep feats seg"]]

for row, arrs in enumerate(arrs):
    for col, arr in enumerate(arrs):
        ax = fig.add_subplot(gs[row + 1, col])
        ax.imshow(arr)
        ax.set_title(titles[row][col])
        ax.axis('off')